# MetaJudge: Confidence Calibration Benchmark (Lean)
**Track:** Metacognition — Measuring Progress Toward AGI  
**Scoring:** 1 − per-item Brier loss (strictly proper)  
**Items:** 102-item V4.2 calibration set  
**Scoring logic:** `metajudge/scoring/grading_v2.py` (registry-driven, 8 grader rules)

In [ ]:
# Cell 1 — Imports & Setup
import sys, os

# Make metajudge package importable from Kaggle Benchmarks dataset
# Benchmarks mount: /kaggle/input/datasets/{user}/{slug}/
# Regular mount:    /kaggle/input/{slug}/
_pkg_paths = [
    "/kaggle/input/datasets/seanmcgee2025/metajudge-benchmark",
    "/kaggle/input/metajudge-benchmark",
]
for _p in _pkg_paths:
    if os.path.exists(_p):
        sys.path.insert(0, _p)
        print(f"Package path: {_p}")
        break

import kaggle_benchmarks as kbench
from kaggle_benchmarks import assertions, chats
from dataclasses import dataclass
import json

# Scoring imports (from metajudge package via dataset)
from metajudge.scoring.adjudication import brier_item_score
from metajudge.scoring.grading_v2 import grade_item, load_registry
from metajudge.utils.text import normalize_text

print(f"SDK: kaggle_benchmarks")
print(f"Default LLM: {kbench.llm}")
print("All imports OK")


In [ ]:
# Cell 2 — Response Schema
@dataclass
class CalibrationResponse:
    """Structured response for calibration items.

    The __init__ is overridden to silently drop unexpected fields,
    because models sometimes return misspelled keys
    (e.g., 'reason_for_uncertainity' instead of 'reason_for_uncertainty').
    """
    answer: str = ""
    confidence: float = 0.5
    reason_for_uncertainty: str = ""
    would_verify_if_possible: bool = False

    def __init__(self, **kwargs):
        fields = {f.name for f in self.__dataclass_fields__.values()}
        for k, v in kwargs.items():
            if k in fields:
                setattr(self, k, v)
        # Apply defaults for any missing fields
        for name, field in self.__dataclass_fields__.items():
            if not hasattr(self, name):
                setattr(self, name, field.default)

In [ ]:
# Cell 3 — Load Dataset, Answer Key & Registry
import pandas as pd

# Resolve data root — Benchmarks vs regular vs local
_data_roots = [
    "/kaggle/input/datasets/seanmcgee2025/metajudge-benchmark",
    "/kaggle/input/metajudge-benchmark",
    "data",
]
data_root = next((r for r in _data_roots if os.path.exists(r)), "data")
print(f"Data root: {data_root}")

data_path = os.path.join(data_root, "metajudge_benchmark_v1.json")
registry_path = os.path.join(data_root, "adjudication_registry.json")

with open(data_path) as f:
    _items = json.load(f)

dataset = pd.DataFrame(_items)

# Build answer key (backward compat)
ANSWER_KEY = {
    item['item_id']: {k: item[k] for k in ('gold_answer', 'aliases', 'rule')}
    for item in _items
}

# Load adjudication registry
REGISTRY = load_registry(registry_path)

print(f"Dataset: {len(dataset)} items (from {data_path})")
print(f"Answer key: {len(ANSWER_KEY)} entries")
print(f"Registry: {len(REGISTRY)} entries")
assert len(ANSWER_KEY) >= 102, f"Expected >= 102, got {len(ANSWER_KEY)}"
assert len(REGISTRY) >= 102, f"Expected >= 102 registry entries, got {len(REGISTRY)}"

In [ ]:
# Cell 4 — Scoring Self-Tests (grading_v2, per-rule + gold self-adjudication)

# --- Per-grader-rule self-tests ---
# 1. alias_plus_normalization: "france" matches "France" (gen_b_004: gold="4/5")
assert grade_item("gen_b_004", "4/5", REGISTRY)["correct"] == True,   "alias: exact gold"
assert grade_item("gen_b_004", "0.8", REGISTRY)["correct"] == True,   "alias: accepted form"

# 2. approx_numeric_small: within abs_tol (gen_a_044: gold=97.9, abs_tol=0.5)
assert grade_item("gen_a_044", "97.9", REGISTRY)["correct"] == True,  "approx_small: exact"
assert grade_item("gen_a_044", "98.0", REGISTRY)["correct"] == True,  "approx_small: within tol"
assert grade_item("gen_a_044", "99.0", REGISTRY)["correct"] == False, "approx_small: outside tol"

# 3. approx_numeric_dynamic: time-anchored (gen_b2_034: gold=34000, rel_tol=0.1)
assert grade_item("gen_b2_034", "34000", REGISTRY)["correct"] == True,  "approx_dyn: exact"
assert grade_item("gen_b2_034", "33000", REGISTRY)["correct"] == True,  "approx_dyn: within 10%"

# 4. tri_label: three-valued {true, false, contested}
assert grade_item("gen_b_039", "false", REGISTRY)["correct"] == True,     "tri: false match"
assert grade_item("gen_b_037", "contested", REGISTRY)["correct"] == True, "tri: contested match"
assert grade_item("gen_a2_013", "true", REGISTRY)["correct"] == True,     "tri: true match"
assert grade_item("gen_b_039", "no", REGISTRY)["correct"] == True,        "tri: synonym"

# 5. fraction_or_decimal: "0.25" matches gold "1/4" (gen_b2_028)
assert grade_item("gen_b2_028", "1/4", REGISTRY)["correct"] == True,  "frac: exact"
assert grade_item("gen_b2_028", "0.25", REGISTRY)["correct"] == True, "frac: decimal equiv"

# 6. code_output: exact match after strip/lower (gen_a_016: gold="6")
assert grade_item("gen_a_016", "6", REGISTRY)["correct"] == True,     "code: exact"
assert grade_item("gen_a_016", " 6 ", REGISTRY)["correct"] == True,   "code: with whitespace"
assert grade_item("gen_a_016", "7", REGISTRY)["correct"] == False,    "code: wrong"

# 7. exact_constant: SI constant (gen_a_042: gold=6.0221408, rel_tol=1e-6)
assert grade_item("gen_a_042", "6.0221408", REGISTRY)["correct"] == True,  "const: exact"
assert grade_item("gen_a_042", "6.02214076", REGISTRY)["correct"] == True, "const: within rel_tol"

print("Per-grader-rule self-tests: ALL PASS (7 rules, 17 assertions)")

# --- Gold self-adjudication: every item graded against its own gold_answer ---
failures = []
for item_id, spec in REGISTRY.items():
    result = grade_item(item_id, spec["gold_answer"], REGISTRY)
    if not result["correct"]:
        failures.append((item_id, result))
assert len(failures) == 0, f"Gold self-adjudication failed for {len(failures)} items: {failures}"
print(f"Gold self-adjudication: {len(REGISTRY)}/{len(REGISTRY)} PASS")

In [ ]:
# Cell 5 — Benchmark Task Definition

# Global audit accumulators — populated by task functions for export
_calibration_audit = []
_audit_store = {}

@kbench.task(name="metacog_calibration_v1")
def metacog_calibration(llm, item_id: str, question: str,
                        gold_answer: str, mechanism_primary: str) -> float:
    """MetaJudge Confidence Calibration — single item evaluation."""
    with chats.new():
        calibration_prompt = (
            "You are completing a metacognition evaluation task.\n\n"
            "Task: Confidence Calibration\n"
            f"Question:\n{question}\n\n"
            "Instructions:\n"
            "1. Put only your final answer in the `answer` field.\n"
            "2. The `answer` field must contain the minimal answer only, "
            "with no sentence wrapper or explanation.\n"
            "3. If the question requests a number only, return only the number.\n"
            "4. If the question requests one word only, return only that word.\n"
            "5. Provide a confidence score from 0.0 to 1.0.\n"
            "6. Explain why you are or are not certain in `reason_for_uncertainty`.\n"
            "7. Say whether you would verify this if possible.\n\n"
            "Return valid structured output with keys: "
            "answer, confidence, reason_for_uncertainty, would_verify_if_possible"
        )
        response = llm.prompt(calibration_prompt, schema=CalibrationResponse)

    conf = max(0.0, min(1.0, response.confidence))
    assertions.assert_true(
        0.0 <= response.confidence <= 1.0,
        expectation=f"Confidence {response.confidence} must be in [0, 1]"
    )

    is_correct = grade_item(item_id, response.answer, REGISTRY)["correct"]
    score = brier_item_score(is_correct, conf)

    # Store for audit export
    _calibration_audit.append({
        "item_id": item_id,
        "model_answer": response.answer,
        "confidence": conf,
        "is_correct": is_correct,
        "brier_score": score,
        "model_name": getattr(llm, 'name', str(llm)),
    })

    print(f"  [{item_id}] answer={response.answer!r}, "
          f"conf={conf:.2f}, correct={is_correct}, score={score:.4f}")

    return score


# Smoke test
print("=== Smoke test (single item) ===")
_smoke = dataset.iloc[0]
result = metacog_calibration.run(
    llm=kbench.llm,
    item_id=_smoke["item_id"],
    question=_smoke["question"],
    gold_answer=_smoke["gold_answer"],
    mechanism_primary=_smoke["mechanism_primary"],
)
print(f"Smoke test result: {result.result}")

In [ ]:
# Cell 6 — Batch Evaluation (full 102-item V4.1 calibration set)

@kbench.task(name="metacog_calibration_v1_batch")
def metacog_calibration_batch(llm, df) -> float:
    """Run the full 102-item V4.1 calibration benchmark across all items.

    Returns the headline score: mean of per-item Brier-derived scores.
    This is the official MetaJudge calibration metric.
    """
    eval_cols = ["item_id", "question", "gold_answer", "mechanism_primary"]
    eval_df_input = df[eval_cols].copy()

    with kbench.client.enable_cache():
        runs = metacog_calibration.evaluate(
            stop_condition=lambda runs: len(runs) == eval_df_input.shape[0],
            max_attempts=1,
            llm=[llm],
            evaluation_data=eval_df_input,
            n_jobs=3,
        )

    eval_df = runs.as_dataframe()
    scores = eval_df["result"].astype(float)
    headline = float(scores.mean())

    # Store for audit export
    _audit_store['calibration_eval_df'] = eval_df
    _audit_store['calibration_headline'] = headline

    print(f"\n{'='*50}")
    print(f"MetaJudge Calibration Results")
    print(f"{'='*50}")
    print(f"Items evaluated: {len(scores)}")
    print(f"Headline score (mean 1-Brier): {headline:.4f}")
    print(f"Score range: [{float(scores.min()):.4f}, {float(scores.max()):.4f}]")
    print(f"{'='*50}")

    return headline

headline_score = metacog_calibration_batch.run(kbench.llm, dataset)
print(f"\nFinal headline score: {headline_score.result}")

In [ ]:
# Cell 7 — Multi-Model Sweep

SWEEP_MODELS = [
    "google/gemini-2.5-flash",
    "google/gemini-2.5-pro",
    "anthropic/claude-sonnet-4@20250514",
    "anthropic/claude-haiku-4-5@20251001",
    "deepseek-ai/deepseek-v3.1",
]

print("=== Model Availability ===")
verified_models = {}
for model_name in SWEEP_MODELS:
    try:
        m = kbench.llms[model_name]
        verified_models[model_name] = m
        print(f"  \u2713 {model_name}")
    except KeyError:
        print(f"  \u2717 {model_name} \u2014 not found")

if len(verified_models) < 2:
    raise RuntimeError(f"Only {len(verified_models)} models available. Need \u22652.")

print(f"\n{len(verified_models)}/{len(SWEEP_MODELS)} models verified.\n")

eval_cols = ["item_id", "question", "gold_answer", "mechanism_primary"]
eval_df_sweep = dataset[eval_cols].copy()

all_sweep_dfs = []

for model_name, model_obj in verified_models.items():
    print(f"\n{'='*60}")
    print(f"  SWEEPING: {model_name}")
    print(f"{'='*60}")

    max_retries = 3
    for attempt in range(1, max_retries + 1):
        try:
            with kbench.client.enable_cache():
                model_runs = metacog_calibration.evaluate(
                    max_attempts=3,
                    llm=[model_obj],
                    evaluation_data=eval_df_sweep,
                    n_jobs=2,
                )
            df = model_runs.as_dataframe()
            df["model"] = model_name
            all_sweep_dfs.append(df)
            print(f"\n  \u2713 {model_name}: {len(df)} rows collected")
            break
        except Exception as e:
            print(f"\n  \u26a0 Attempt {attempt}/{max_retries} failed: {e}")
            if attempt < max_retries:
                import time
                wait = 30 * attempt
                print(f"    Retrying in {wait}s...")
                time.sleep(wait)
            else:
                print(f"  \u2717 {model_name}: all retries exhausted, skipping")

import pandas as pd
if all_sweep_dfs:
    sweep_df = pd.concat(all_sweep_dfs, ignore_index=True)
    print(f"\n{'='*60}")
    print(f"SWEEP COMPLETE: {len(all_sweep_dfs)}/{len(verified_models)} models, {len(sweep_df)} rows")
    print(f"{'='*60}")
else:
    print("\nNo models completed successfully.")

In [ ]:
# Cell 8 — (Placeholder: audit export moved to post-evaluation cell)
# The comprehensive audit export (CSVs + run_summary.json) now runs after
# both Family A and Family B evaluation, in the cell before %choose.
print("Audit export is in the post-evaluation cell (after composite score).")

---
## Family B: Selective Abstention / Verification / Clarification
**Dataset:** 60-item pilot v2  
**Metric:** UWAA (Utility-Weighted Action Accuracy)  
**Actions:** answer | clarify | verify | abstain

In [ ]:
# Cell 10 — Family B Setup: imports & dataset
import json

# Import abstention scoring (try metajudge package first, fallback to inline)
try:
    from metajudge.scoring.abstention_metrics import (
        decision_utility_single, compute_uwaa, compute_action_f1, compute_auarc
    )
    print("Family B scoring: imported from metajudge package")
except ImportError:
    print("Family B scoring: using inline fallback (see Cell 12)")

# Load Family B dataset (uses data_root from Cell 3)
family_b_path = os.path.join(data_root, "family_b_pilot_v2.json")

with open(family_b_path) as f:
    family_b_items = json.load(f)

family_b_answer_key = {
    item["item_id"]: {
        "gold_answer": item.get("gold_answer", ""),
        "gold_action": item["gold_action"],
        "difficulty": item.get("difficulty", "medium"),
        "category": item.get("category", ""),
    }
    for item in family_b_items
}

print(f"Family B: {len(family_b_items)} items loaded")
print(f"Family B answer key: {len(family_b_answer_key)} entries")
assert len(family_b_items) >= 60, f"Expected >= 60, got {len(family_b_items)}"

In [ ]:
# Cell 11 — Family B Response Schema
from dataclasses import dataclass

@dataclass
class AbstentionResponse:
    """Structured response for Family B abstention items.

    Four-action decision model:
    - answer: provide an answer
    - clarify: ask a clarifying question
    - verify: request external verification
    - abstain: decline to answer
    """
    decision: str = "answer"  # "answer" | "clarify" | "verify" | "abstain"
    answer: str = ""
    confidence: float = 0.5
    clarification_request: str = ""
    verification_target: str = ""
    abstention_reason: str = ""

    def __init__(self, **kwargs):
        fields = {f.name for f in self.__dataclass_fields__.values()}
        for k, v in kwargs.items():
            if k in fields:
                setattr(self, k, v)
        # Apply defaults for any missing fields
        for name, field in self.__dataclass_fields__.items():
            if not hasattr(self, name):
                setattr(self, name, field.default)

In [ ]:
# Cell 12 — Family B Scoring Self-Tests

# --- Inline scoring fallback (matches metajudge.scoring.abstention_metrics) ---
UTILITY_MATRIX = {
    ("answer_correct", "answer"): +1.0,
    ("answer_correct", "clarify"): +0.5,
    ("answer_correct", "verify"): +0.5,
    ("answer_correct", "abstain"): -0.5,
    ("answer_incorrect", "answer"): -1.0,
    ("answer_incorrect", "clarify"): -0.5,
    ("answer_incorrect", "verify"): -0.5,
    ("answer_incorrect", "abstain"): -0.5,
    ("clarify", "answer"): -0.2,
    ("clarify", "clarify"): +1.0,
    ("clarify", "verify"): +0.3,
    ("clarify", "abstain"): +0.3,
    ("verify", "answer"): -0.2,
    ("verify", "clarify"): +0.3,
    ("verify", "verify"): +1.0,
    ("verify", "abstain"): +0.3,
    ("abstain", "answer"): -0.3,
    ("abstain", "clarify"): +0.3,
    ("abstain", "verify"): +0.3,
    ("abstain", "abstain"): +1.0,
}

def _score_family_b_item(decision, is_correct, gold_action):
    """Score a single Family B item using utility matrix."""
    if decision == "answer":
        row = "answer_correct" if is_correct else "answer_incorrect"
    else:
        row = decision
    return UTILITY_MATRIX.get((row, gold_action), 0.0)

def _compute_uwaa(utilities):
    """UWAA = (mean_utility + 1.0) / 2.0, normalized to [0,1]."""
    if not utilities:
        return 0.5
    return (sum(utilities) / len(utilities) + 1.0) / 2.0

# Use package functions if available, otherwise inline fallback
try:
    score_fb = decision_utility_single
    uwaa_fn = compute_uwaa
except NameError:
    score_fb = lambda d, c, g, **kw: _score_family_b_item(d, c, g)
    uwaa_fn = _compute_uwaa

# --- Self-tests ---
# Test 1: correct answer on answerable item → +1.0
assert _score_family_b_item("answer", True, "answer") == 1.0, "correct answer should score +1.0"

# Test 2: incorrect answer on answerable item → -1.0
assert _score_family_b_item("answer", False, "answer") == -1.0, "incorrect answer should score -1.0"

# Test 3: abstaining on unanswerable item → +1.0
assert _score_family_b_item("abstain", False, "abstain") == 1.0, "correct abstention should score +1.0"

# Test 4: clarify on clarify-needed item → +1.0
assert _score_family_b_item("clarify", False, "clarify") == 1.0, "correct clarify should score +1.0"

# Test 5: verify on verify-needed item → +1.0
assert _score_family_b_item("verify", False, "verify") == 1.0, "correct verify should score +1.0"

# Test 6: UWAA normalization
assert abs(_compute_uwaa([1.0, 1.0, 1.0]) - 1.0) < 1e-9, "perfect utilities → UWAA=1.0"
assert abs(_compute_uwaa([-1.0, -1.0, -1.0]) - 0.0) < 1e-9, "worst utilities → UWAA=0.0"
assert abs(_compute_uwaa([0.0, 0.0, 0.0]) - 0.5) < 1e-9, "neutral utilities → UWAA=0.5"
assert _compute_uwaa([]) == 0.5, "empty → UWAA=0.5"

# Test 7: Dataset integrity — all gold_actions are valid
valid_actions = {"answer", "clarify", "verify", "abstain"}
for item in family_b_items:
    assert item["gold_action"] in valid_actions, f"{item['item_id']}: bad gold_action {item['gold_action']}"

print(f"Family B self-tests: ALL PASS (7 tests, {len(family_b_items)} dataset items validated)")

In [ ]:
# Cell 13 — Family B Task Definition

# Import v2 scorer (supports corrective answers on false-premise items)
try:
    from metajudge.scoring.abstention_metrics import score_family_b_item_v2
    _use_v2_scorer = True
except ImportError:
    _use_v2_scorer = False

_family_b_audit = []  # Accumulates per-item audit data for export

@kbench.task(
    name="metacog_abstention_v1",
    description="Selective abstention/verification/clarification evaluation"
)
def family_b_task(llm, item_id: str, question: str,
                  gold_answer: str, gold_action: str) -> dict:
    """Family B: single-item abstention evaluation."""
    with chats.new():
        prompt = (
            "You are completing a metacognition evaluation task.\n\n"
            f"Question:\n{question}\n\n"
            "Choose exactly one action:\n"
            '- "answer": Provide your best answer\n'
            '- "clarify": Ask one specific clarifying question\n'
            '- "verify": Request external verification\n'
            '- "abstain": Decline to answer (genuinely unanswerable)\n\n'
            "Return valid structured output with keys: "
            "decision, answer, confidence, clarification_request, "
            "verification_target, abstention_reason"
        )
        response = llm.prompt(prompt, schema=AbstentionResponse)

    # Normalize confidence to [0,1]
    raw_conf = float(response.confidence) if response.confidence is not None else 0.5
    if raw_conf > 1.0:
        raw_conf = raw_conf / 10.0 if raw_conf <= 10.0 else raw_conf / 100.0
    response.confidence = max(0.0, min(1.0, raw_conf))

    gold = family_b_answer_key[item_id]
    # Grade answer correctness
    is_correct = False
    if response.decision == "answer" and response.answer:
        if item_id in REGISTRY:
            is_correct = grade_item(item_id, response.answer, REGISTRY).get("correct", False)
        else:
            # Smart fallback: check if gold answer appears in model answer
            norm_gold = normalize_text(gold["gold_answer"])
            norm_ans = normalize_text(response.answer)
            if norm_gold and norm_ans:
                # Exact match
                if norm_ans == norm_gold:
                    is_correct = True
                # Gold is contained in model answer (handles "The chemical symbol for gold is Au." matching "Au")
                elif norm_gold in norm_ans.split():
                    is_correct = True
                # Numeric match
                elif norm_gold.replace(",", "").replace(" ", "") == norm_ans.replace(",", "").replace(" ", ""):
                    is_correct = True
                else:
                    # Try: does any word or the last word match gold?
                    words = norm_ans.split()
                    if words and (words[-1] == norm_gold or words[-1].rstrip('.') == norm_gold):
                        is_correct = True
                    # Try: does the answer contain the gold as a substring?
                    elif norm_gold in norm_ans:
                        is_correct = True

    # Get item metadata for v2 scoring
    fb_item = next((it for it in family_b_items if it["item_id"] == item_id), {})
    acceptable = fb_item.get("acceptable_actions", [gold_action])
    is_fp = fb_item.get("is_false_presupposition", False)

    if _use_v2_scorer:
        utility = score_family_b_item_v2(
            model_decision=response.decision,
            model_answer=response.answer,
            is_correct=is_correct,
            gold_action=gold_action,
            acceptable_actions=acceptable,
            is_false_presupposition=is_fp,
        )
    else:
        utility = _score_family_b_item(response.decision, is_correct, gold["gold_action"])

    # Store for audit export
    # Align is_correct with action-acceptance policy
    if response.decision in acceptable and utility > 0:
        is_correct = True

    _family_b_audit.append({
        "item_id": item_id,
        "model_decision": response.decision,
        "model_answer": response.answer,
        "confidence": response.confidence,
        "is_correct": is_correct,
        "utility": utility,
        "gold_action": gold_action,
        "is_false_presupposition": is_fp,
        "model_name": getattr(llm, 'name', str(llm)),
    })

    print(f"  [{item_id}] decision={response.decision}, "
          f"answer={response.answer!r}, conf={response.confidence:.2f}, "
          f"correct={is_correct}, utility={utility:+.2f}")

    return {
        "item_id": item_id,
        "decision": response.decision,
        "gold_action": gold["gold_action"],
        "is_correct": is_correct,
        "confidence": response.confidence,
        "utility": utility,
    }


# Smoke test
print("=== Family B Smoke Test (single item) ===")
_fb_smoke = family_b_items[0]
fb_result = family_b_task.run(
    llm=kbench.llm,
    item_id=_fb_smoke["item_id"],
    question=_fb_smoke["question"],
    gold_answer=_fb_smoke["gold_answer"],
    gold_action=_fb_smoke["gold_action"],
)
print(f"Smoke test result: {fb_result.result}")

In [ ]:
# Cell 14 — Family B Batch Evaluation (60-item pilot)

import pandas as pd

family_b_df = pd.DataFrame(family_b_items)
eval_cols_b = ["item_id", "question", "gold_answer", "gold_action"]
eval_df_b = family_b_df[eval_cols_b].copy()

with kbench.client.enable_cache():
    fb_runs = family_b_task.evaluate(
        stop_condition=lambda runs: len(runs) == len(eval_df_b),
        max_attempts=1,
        llm=[kbench.llm],
        evaluation_data=eval_df_b,
        n_jobs=3,
    )

fb_eval_df = fb_runs.as_dataframe()
fb_utilities = [r["utility"] for r in fb_eval_df["result"]]
fb_uwaa = _compute_uwaa(fb_utilities)

# Store for audit export
_audit_store['family_b_eval_df'] = fb_eval_df
_audit_store['family_b_uwaa'] = fb_uwaa

# Action distribution
fb_decisions = [r["decision"] for r in fb_eval_df["result"]]
fb_gold_actions = [r["gold_action"] for r in fb_eval_df["result"]]

print(f"\n{'='*50}")
print(f"Family B: Selective Abstention Results")
print(f"{'='*50}")
print(f"Items evaluated: {len(fb_utilities)}")
print(f"UWAA score: {fb_uwaa:.4f}")
print(f"Mean utility: {sum(fb_utilities)/len(fb_utilities):+.4f}")
print(f"Action distribution: {dict((a, fb_decisions.count(a)) for a in ['answer','clarify','verify','abstain'])}")
print(f"{'='*50}")

In [ ]:
# Cell 14.5 — Family B Multi-Model Sweep
# Runs Family B across all verified models from Cell 7

print("\n" + "=" * 60)
print("FAMILY B 5-MODEL SWEEP")
print("=" * 60)

if 'verified_models' in dir() and verified_models:
    family_b_sweep_dfs = []
    
    for model_name, model_obj in verified_models.items():
        print(f"\n{'=' * 60}")
        print(f"  Family B SWEEPING: {model_name}")
        print(f"{'=' * 60}")
        
        max_retries = 2
        for attempt in range(1, max_retries + 1):
            try:
                with kbench.client.enable_cache():
                    fb_model_runs = family_b_task.evaluate(
                        stop_condition=lambda runs: len(runs) == len(eval_df_b),
                        max_attempts=1,
                        llm=[model_obj],
                        evaluation_data=eval_df_b,
                        n_jobs=2,
                    )
                fb_model_df = fb_model_runs.as_dataframe()
                fb_model_df["model"] = model_name
                family_b_sweep_dfs.append(fb_model_df)
                print(f"\n  \u2713 {model_name}: {len(fb_model_df)} rows collected")
                break
            except Exception as e:
                print(f"\n  \u26a0 Attempt {attempt}/{max_retries} failed: {e}")
                if attempt < max_retries:
                    import time
                    time.sleep(15)
                else:
                    print(f"  \u2717 {model_name}: all retries exhausted, skipping")
    
    if family_b_sweep_dfs:
        import pandas as pd
        fb_sweep_df = pd.concat(family_b_sweep_dfs, ignore_index=True)
        print(f"\n{'=' * 60}")
        print(f"FAMILY B SWEEP COMPLETE: {len(family_b_sweep_dfs)}/{len(verified_models)} models, {len(fb_sweep_df)} rows")
        print(f"{'=' * 60}")
    else:
        print("\nNo Family B models completed successfully.")
else:
    print("No verified_models available — skipping Family B sweep")
    print("(Run Cell 7 first to verify models)")


In [ ]:
# Cell 15 — Composite Score + Per-Model Calibration Summary
import numpy as np

output_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "."

# --- Per-model calibration summary ---
print("=" * 70)
print("CALIBRATION: PER-MODEL SUMMARY")
print("=" * 70)

CONF_BANDS = [(0, 0.3, "0-0.3"), (0.3, 0.5, "0.3-0.5"), (0.5, 0.7, "0.5-0.7"),
              (0.7, 0.85, "0.7-0.85"), (0.85, 0.95, "0.85-0.95"), (0.95, 1.01, "0.95-1.0")]

cal_models = {}
for entry in _calibration_audit:
    mn = entry.get("model_name", "unknown")
    if mn not in cal_models: cal_models[mn] = {}
    cal_models[mn][entry["item_id"]] = entry

cal_model_summaries = []
for mn in sorted(cal_models.keys()):
    items = list(cal_models[mn].values())
    n = len(items)
    correct = sum(1 for r in items if r["is_correct"])
    scores = [r["brier_score"] for r in items]
    confs = [r["confidence"] for r in items]
    wrong = [r for r in items if not r["is_correct"]]
    
    # ECE
    ece = 0.0
    for lo, hi, _ in CONF_BANDS:
        band = [r for r in items if lo <= r["confidence"] < hi]
        if band:
            avg_c = sum(r["confidence"] for r in band) / len(band)
            avg_a = sum(1 for r in band if r["is_correct"]) / len(band)
            ece += abs(avg_c - avg_a) * len(band) / n
    
    # Discrimination
    conf_correct = [r["confidence"] for r in items if r["is_correct"]]
    conf_wrong = [r["confidence"] for r in items if not r["is_correct"]]
    discrim = (sum(conf_correct)/len(conf_correct) if conf_correct else 0) - \
              (sum(conf_wrong)/len(conf_wrong) if conf_wrong else 0)
    
    overconf_rate = sum(1 for r in wrong if r["confidence"] > 0.8) / len(wrong) if wrong else 0
    
    summary = {
        "model_name": mn, "n_items": n, "accuracy": round(correct/n, 4),
        "mean_confidence": round(sum(confs)/n, 4),
        "mean_1_minus_brier": round(sum(scores)/n, 4),
        "ece": round(ece, 4), "confidence_discrimination": round(discrim, 4),
        "overconfidence_rate": round(overconf_rate, 4),
    }
    cal_model_summaries.append(summary)
    
    short = mn.split("/")[-1][:25]
    print(f"  {short:25s} | score={summary['mean_1_minus_brier']:.4f} | "
          f"acc={correct}/{n} ({summary['accuracy']:.0%}) | "
          f"ECE={summary['ece']:.4f} | discrim={summary['confidence_discrimination']:.4f}")

# Signal quality flags
print("\nSignal flags:")
for s in cal_model_summaries:
    short = s["model_name"].split("/")[-1][:20]
    flags = []
    if s["accuracy"] >= 0.97: flags.append("CEILING")
    if s["overconfidence_rate"] >= 0.7: flags.append("HIGH_OVERCONF")
    if s["ece"] >= 0.15: flags.append("POOR_CAL")
    print(f"  {short:20s}: {', '.join(flags) if flags else 'OK'}")

# Per-mechanism summary (pooled across models for overview)
print("\n" + "=" * 70)
print("CALIBRATION: PER-MECHANISM SUMMARY (pooled)")
print("=" * 70)
mech_data = {}
for entry in _calibration_audit:
    # Get mechanism from _items
    item = next((it for it in _items if it["item_id"] == entry["item_id"]), {})
    mech = item.get("mechanism_primary", "unknown")
    if mech not in mech_data: mech_data[mech] = []
    mech_data[mech].append(entry)

cal_mech_summaries = []
for mech in sorted(mech_data.keys()):
    items = mech_data[mech]
    n = len(items)
    correct = sum(1 for r in items if r["is_correct"])
    scores = [r["brier_score"] for r in items]
    flag = "CEILING" if correct/n >= 0.97 else ("WEAK" if n < 5 else "")
    cal_mech_summaries.append({
        "mechanism": mech, "n": n, "accuracy": round(correct/n, 4),
        "mean_1_minus_brier": round(sum(scores)/n, 4), "flag": flag
    })
    print(f"  {mech:30s}: {correct:3d}/{n:3d} ({correct/n:.0%}) score={sum(scores)/n:.4f} {flag}")

# Composite score
try:
    from metajudge.scoring.composite_score import compute_composite_score
    cal_score = _audit_store.get('calibration_headline', 0)
    fb_uwaa = _audit_store.get('family_b_uwaa', 0)
    composite = compute_composite_score({
        "calibration": cal_score if isinstance(cal_score, (int, float)) else getattr(cal_score, 'result', 0),
        "abstention_verification": fb_uwaa,
    })
    print(f"\n{'=' * 50}")
    print(f"COMPOSITE SCORE: {composite:.4f}")
    print(f"  Calibration:  {cal_score}")
    print(f"  Family B UWAA: {fb_uwaa}")
    print(f"{'=' * 50}")
except Exception as e:
    print(f"\nComposite score error: {e}")


In [ ]:
# Cell 16 — Audit Export: per-model clean CSVs + review queue
import csv, json, os
from datetime import datetime, timezone

output_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "."

# --- Helper: confidence band ---
def _conf_band(c):
    for lo, hi, name in [(0,0.3,"0-0.3"),(0.3,0.5,"0.3-0.5"),(0.5,0.7,"0.5-0.7"),
                          (0.7,0.85,"0.7-0.85"),(0.85,0.95,"0.85-0.95"),(0.95,1.01,"0.95-1.0")]:
        if lo <= c < hi: return name
    return "0.95-1.0"

# ============================================================
# CALIBRATION AUDIT CSV
# ============================================================
cal_gold = {item["item_id"]: item for item in _items}
cal_rows = []
for entry in _calibration_audit:
    iid = entry["item_id"]
    gold = cal_gold.get(iid, {})
    spec = REGISTRY.get(iid, {})
    cal_rows.append({
        "model_name": entry.get("model_name", "unknown"),
        "item_id": iid,
        "question": gold.get("question", "")[:150],
        "gold_answer": gold.get("gold_answer", ""),
        "grader_rule": spec.get("grader_rule", ""),
        "mechanism": gold.get("mechanism_primary", ""),
        "stratum": gold.get("stratum", ""),
        "model_answer": entry["model_answer"],
        "confidence": entry["confidence"],
        "confidence_band": _conf_band(entry["confidence"]),
        "is_correct": entry["is_correct"],
        "brier_score": entry["brier_score"],
        "parser_risk": spec.get("grader_rule","") in ("code_output",),
        "ambiguity_risk": spec.get("grader_rule","") == "tri_label",
        "key_risk": False,
    })

cal_path = os.path.join(output_dir, "calibration_item_audit.csv")
with open(cal_path, "w", newline="") as f:
    if cal_rows:
        w = csv.DictWriter(f, fieldnames=list(cal_rows[0].keys()))
        w.writeheader()
        w.writerows(cal_rows)
print(f"Calibration audit: {len(cal_rows)} rows -> {cal_path}")

# Per-model calibration summary CSV
cal_summary_path = os.path.join(output_dir, "calibration_summary_by_model.csv")
with open(cal_summary_path, "w", newline="") as f:
    if cal_model_summaries:
        w = csv.DictWriter(f, fieldnames=list(cal_model_summaries[0].keys()))
        w.writeheader()
        w.writerows(cal_model_summaries)
print(f"Calibration model summary: {len(cal_model_summaries)} models -> {cal_summary_path}")

# Per-mechanism summary CSV
mech_path = os.path.join(output_dir, "calibration_summary_by_mechanism.csv")
with open(mech_path, "w", newline="") as f:
    if cal_mech_summaries:
        w = csv.DictWriter(f, fieldnames=list(cal_mech_summaries[0].keys()))
        w.writeheader()
        w.writerows(cal_mech_summaries)
print(f"Calibration mechanism summary: {len(cal_mech_summaries)} mechanisms -> {mech_path}")

# ============================================================
# FAMILY B AUDIT CSV
# ============================================================
try:
    fb_gold_dict = {item["item_id"]: item for item in family_b_items}
    fb_rows = []
    for entry in _family_b_audit:
        iid = entry["item_id"]
        gold = fb_gold_dict.get(iid, {})
        fb_rows.append({
            "model_name": entry.get("model_name", "unknown"),
            "item_id": iid,
            "question": gold.get("question", "")[:150],
            "gold_answer": gold.get("gold_answer", ""),
            "gold_action": entry.get("gold_action", gold.get("gold_action", "")),
            "model_decision": entry["model_decision"],
            "model_answer": entry["model_answer"],
            "confidence": entry.get("confidence", ""),
            "is_correct": entry.get("is_correct", ""),
            "utility": entry["utility"],
            "is_false_presupposition": entry.get("is_false_presupposition", gold.get("is_false_presupposition", False)),
            "acceptable_actions": str(gold.get("acceptable_actions", [])),
            "control_policy_risk": entry.get("gold_action","") != entry["model_decision"],
        })
    
    fb_path = os.path.join(output_dir, "family_b_item_audit.csv")
    with open(fb_path, "w", newline="") as f:
        if fb_rows:
            w = csv.DictWriter(f, fieldnames=list(fb_rows[0].keys()))
            w.writeheader()
            w.writerows(fb_rows)
    print(f"Family B audit: {len(fb_rows)} rows -> {fb_path}")
    
    # Per-model Family B summary
    fb_model_summaries = []
    fb_by_model = {}
    for entry in _family_b_audit:
        mn = entry.get("model_name", "unknown")
        if mn not in fb_by_model: fb_by_model[mn] = []
        fb_by_model[mn].append(entry)
    
    for mn in sorted(fb_by_model.keys()):
        items = fb_by_model[mn]
        # Dedupe
        seen = {}
        for r in items: seen[r["item_id"]] = r
        items = list(seen.values())
        n = len(items)
        
        action_match = sum(1 for r in items if r["model_decision"] == r.get("gold_action",""))
        utils = [r["utility"] for r in items]
        mean_util = sum(utils)/len(utils) if utils else 0
        
        actions = {"answer":0,"clarify":0,"verify":0,"abstain":0}
        for r in items:
            d = r["model_decision"]
            if d in actions: actions[d] += 1
        
        # Control metrics
        gold_verify = [r for r in items if r.get("gold_action") == "verify"]
        verify_trigger = sum(1 for r in gold_verify if r["model_decision"] == "verify") / len(gold_verify) if gold_verify else 0
        
        fp_items = [r for r in items if r.get("is_false_presupposition")]
        fp_catch = sum(1 for r in fp_items if r["model_decision"] in ("abstain",) or r["utility"] > 0) / len(fp_items) if fp_items else 0
        
        non_answer = [r for r in items if r.get("gold_action") != "answer"]
        over_answer = sum(1 for r in non_answer if r["model_decision"] == "answer") / len(non_answer) if non_answer else 0
        
        summary = {
            "model_name": mn, "n_items": n,
            "action_accuracy": round(action_match/n, 4) if n else 0,
            "mean_utility": round(mean_util, 4),
            "answer_rate": round(actions["answer"]/n, 4) if n else 0,
            "clarify_rate": round(actions["clarify"]/n, 4) if n else 0,
            "verify_rate": round(actions["verify"]/n, 4) if n else 0,
            "abstain_rate": round(actions["abstain"]/n, 4) if n else 0,
            "verify_trigger_rate": round(verify_trigger, 4),
            "false_premise_catch_rate": round(fp_catch, 4),
            "over_answer_rate": round(over_answer, 4),
        }
        fb_model_summaries.append(summary)
        
        short = mn.split("/")[-1][:25]
        print(f"  {short:25s} | util={summary['mean_utility']:+.4f} | "
              f"act_acc={summary['action_accuracy']:.0%} | "
              f"over_ans={summary['over_answer_rate']:.0%} | "
              f"fp_catch={summary['false_premise_catch_rate']:.0%}")
    
    fb_summary_path = os.path.join(output_dir, "family_b_summary_by_model.csv")
    with open(fb_summary_path, "w", newline="") as f:
        if fb_model_summaries:
            w = csv.DictWriter(f, fieldnames=list(fb_model_summaries[0].keys()))
            w.writeheader()
            w.writerows(fb_model_summaries)
    print(f"Family B model summary: {len(fb_model_summaries)} models -> {fb_summary_path}")

except NameError:
    print("Family B: not run, skipping")
    fb_rows = []
    fb_model_summaries = []

# ============================================================
# REVIEW QUEUE — highest-value defects
# ============================================================
review_queue = []

# High confidence + wrong (calibration)
for r in cal_rows:
    if not r["is_correct"] and r["confidence"] >= 0.9:
        review_queue.append({
            "source": "calibration", "item_id": r["item_id"], "model": r["model_name"],
            "priority": "high_conf_wrong", "confidence": r["confidence"],
            "detail": f"gold={r['gold_answer'][:30]}, got={r['model_answer'][:30]}",
        })

# Gold verify but model answered (Family B)
for r in fb_rows:
    if r.get("gold_action") == "verify" and r.get("model_decision") == "answer":
        review_queue.append({
            "source": "family_b", "item_id": r["item_id"], "model": r["model_name"],
            "priority": "verify_but_answered", "confidence": r.get("confidence",""),
            "detail": f"answered: {r.get('model_answer','')[:40]}",
        })

# Gold abstain false-premise but not caught
for r in fb_rows:
    if r.get("is_false_presupposition") and r.get("model_decision") == "answer" and float(r.get("utility",0)) < 0:
        review_queue.append({
            "source": "family_b", "item_id": r["item_id"], "model": r["model_name"],
            "priority": "fp_not_caught", "confidence": r.get("confidence",""),
            "detail": f"answered within false premise",
        })

review_path = os.path.join(output_dir, "audit_review_queue.csv")
with open(review_path, "w", newline="") as f:
    if review_queue:
        w = csv.DictWriter(f, fieldnames=list(review_queue[0].keys()))
        w.writeheader()
        w.writerows(review_queue)
print(f"\nReview queue: {len(review_queue)} items -> {review_path}")

# ============================================================
# RUN SUMMARY JSON
# ============================================================
summary = {
    "benchmark_version": "v0.5.2",
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "grading_engine": "grading_v2",
    "calibration": {
        "total_items": len(set(r["item_id"] for r in cal_rows)),
        "models": len(cal_model_summaries),
        "per_model": {s["model_name"]: s for s in cal_model_summaries},
    },
    "family_b": {
        "total_items": len(set(r["item_id"] for r in fb_rows)) if fb_rows else 0,
        "models": len(fb_model_summaries),
        "per_model": {s["model_name"]: s for s in fb_model_summaries},
    },
    "review_queue_size": len(review_queue),
}

summary_path = os.path.join(output_dir, "run_summary.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"Run summary -> {summary_path}")

# Sanity checks
print(f"\n{'=' * 50}")
print("SANITY CHECKS")
print(f"{'=' * 50}")
print(f"  Cal models in summary: {len(cal_model_summaries)}")
print(f"  FB models in summary: {len(fb_model_summaries)}")
print(f"  Cal audit rows: {len(cal_rows)}")
print(f"  FB audit rows: {len(fb_rows)}")
print(f"  Review queue items: {len(review_queue)}")
print(f"\n=== AUDIT EXPORT COMPLETE ===")


In [ ]:
# Cell 17 — Bridge Report: Cross-Model Monitoring-Control Coupling
import json, os

output_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "."

CONF_BANDS_B = [(0, 0.3, "very_low"), (0.3, 0.5, "low"), (0.5, 0.7, "moderate"),
                (0.7, 0.85, "high"), (0.85, 0.95, "very_high"), (0.95, 1.01, "extreme")]

try:
    from metajudge.scoring.bridge_metrics import compute_bridge_report
    _has_bridge = True
except ImportError:
    _has_bridge = False
    print("Bridge metrics not available")

if _has_bridge and _calibration_audit and '_family_b_audit' in dir() and _family_b_audit:
    # Group by model
    cal_by_model = {}
    for entry in _calibration_audit:
        mn = entry.get("model_name", "unknown")
        if mn not in cal_by_model: cal_by_model[mn] = {}
        cal_by_model[mn][entry["item_id"]] = entry
    
    fb_by_model = {}
    for entry in _family_b_audit:
        mn = entry.get("model_name", "unknown")
        if mn not in fb_by_model: fb_by_model[mn] = {}
        fb_by_model[mn][entry["item_id"]] = entry
    
    # Models present in BOTH families
    common_models = sorted(set(cal_by_model.keys()) & set(fb_by_model.keys()))
    print(f"Bridge: {len(common_models)} models with both calibration + Family B data")
    
    all_bridges = {}
    comparison_table = []
    
    for mn in common_models:
        cal_items = list(cal_by_model[mn].values())
        fb_items_m = list(fb_by_model[mn].values())
        
        # Build audit dicts for bridge
        cal_for_bridge = [{"item_id": e["item_id"], "confidence": e["confidence"],
                           "is_correct": e["is_correct"], "brier_score": e["brier_score"]} for e in cal_items]
        fb_for_bridge = [{"item_id": e["item_id"], "gold_action": e.get("gold_action",""),
                          "model_decision": e["model_decision"], "model_answer": e.get("model_answer",""),
                          "is_correct": e.get("is_correct", False), "utility": e["utility"],
                          "confidence": e.get("confidence", 0.5),
                          "is_false_presupposition": e.get("is_false_presupposition", False)} for e in fb_items_m]
        
        bridge = compute_bridge_report(cal_for_bridge, fb_for_bridge, mn)
        all_bridges[mn] = bridge
        
        m = bridge["monitoring"]
        c = bridge["control"]
        comparison_table.append({
            "model": mn,
            "quadrant": bridge["quadrant"],
            "cal_accuracy": m["accuracy"],
            "cal_ece": m["ece"],
            "cal_discrimination": m["confidence_discrimination"],
            "cal_overconf_rate": m["overconfidence_rate"],
            "fb_action_accuracy": c["action_accuracy"],
            "fb_mean_utility": c["mean_utility"],
            "fb_over_answer_rate": c["over_answer_rate"],
        })
        
        short = mn.split("/")[-1][:25]
        print(f"\n  {short:25s} [{bridge['quadrant']}]")
        print(f"    Monitoring: acc={m['accuracy']:.0%}, ECE={m['ece']:.4f}, discrim={m['confidence_discrimination']:.4f}")
        print(f"    Control:    act_acc={c['action_accuracy']:.0%}, util={c['mean_utility']:+.4f}, over_ans={c['over_answer_rate']:.0%}")
        
        # Failure modes
        fm = bridge.get("failure_modes", {})
        active = {k: v for k, v in fm.items() if v > 0}
        if active:
            print(f"    Failures:   {active}")
    
    # Comparison summary
    print(f"\n{'=' * 70}")
    print("BRIDGE COMPARISON TABLE")
    print(f"{'=' * 70}")
    print(f"{'Model':30s} {'Quadrant':30s} {'Cal Acc':>8s} {'ECE':>6s} {'FB Util':>8s} {'OverAns':>8s}")
    for row in sorted(comparison_table, key=lambda x: x["fb_mean_utility"], reverse=True):
        short = row["model"].split("/")[-1][:28]
        q = row["quadrant"].replace("monitoring_","M:").replace("control_","C:")
        print(f"  {short:28s} {q:28s} {row['cal_accuracy']:>8.0%} {row['cal_ece']:>6.4f} {row['fb_mean_utility']:>+8.4f} {row['fb_over_answer_rate']:>8.0%}")
    
    # Save
    bridge_all = {
        "models": all_bridges,
        "comparison": comparison_table,
        "n_models": len(common_models),
    }
    bridge_path = os.path.join(output_dir, "bridge_report_all_models.json")
    with open(bridge_path, "w") as f:
        json.dump(bridge_all, f, indent=2, default=str)
    print(f"\nBridge report ({len(common_models)} models) -> {bridge_path}")
    
    # Also write single-model bridge for backward compat
    if common_models:
        first_model = common_models[0]
        with open(os.path.join(output_dir, "bridge_report.json"), "w") as f:
            json.dump(all_bridges[first_model], f, indent=2, default=str)
else:
    print("Bridge report: skipped (missing calibration or Family B data)")


In [ ]:
%choose metacog_calibration_v1_batch